# Piloto 2026 · aromas con transferencia gas–líquido

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "fermentation_model").exists():
    ROOT = ROOT.parent
if not (ROOT / "fermentation_model").exists():
    raise RuntimeError("Execute from the repository or a descendant directory")
import os
sys.path.insert(0, str(ROOT / "fermentation_model"))
from pilot_2026 import run_aroma_co2_release_recalibration_2026 as analysis
result = analysis.load_results() if os.environ.get("PILOT_AROMA_REUSE_RESULTS") == "1" else analysis.run_analysis()
print("Resultados:", analysis.RESULTS_DIR.relative_to(ROOT))

## tl;dr

In [ ]:
summary = result["summary"]
display(summary["holdout_rmse_comparison"])
print("Mediana de recambios gaseosos acumulados:", round(summary["median_integrated_gas_turnover"], 4))
print("Pérdida identificada:", summary["loss_identified_with_dynamic_transfer"])
print("26157 MIX-03:", summary["ethyl_octanoate_26157_mix03"])
print("Gate:", result["gate"]["verdict"])

## Contexto y métodos

- Se conservan formación aromática, partición UNIFAC, captura, errores y censura.
- Ambos comparadores usan el mismo rCO₂ emitido predicho por `solubility_o2_nitrogen_boost_continuous_release`.
- La corrección convierte masa de CO₂ a `Qgas/VL` con gas ideal y usa `1-exp(-kLa(E)/(Qgas/VL))`.
- `kLa(E)=kLa_ref*mE**((E-50)/10)` permite contrastar directamente el efecto adicional del etanol.
- `K(T, etanol, azúcar)` sigue aportando la dependencia termodinámica con la composición.
- `26158` y `26211` son holdouts tanto para CO₂ como para aromas.
- La primera muestra de vino inicializa la trayectoria y no entra al RMSE de validación; las muestras siguientes sí.
- Se estiman `mass_transfer_kla_ref_h_inv` y el multiplicador de etanol, con eficiencia NTU acotada entre cero y uno.

## Datos y forcing rCO₂

In [ ]:
display(result["exposure"].round(4))
display(Image(filename=analysis.FIGURE_DIR / "gas_flow_conversion.png"))
display(Image(filename=analysis.FIGURE_DIR / "integrated_rco2_exposure.png"))

## Resultados de calibración y validación

In [ ]:
display(result["parameters"].query("fit_scope == 'all_data'").round(5))
display(result["metrics"].query("fit_scope == 'calibration_only' and role == 'holdout'").round(4))
display(Image(filename=analysis.FIGURE_DIR / "holdout_wine_predictions.png"))
display(Image(filename=analysis.FIGURE_DIR / "holdout_wine_rmse_comparison.png"))

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / "holdout_condensate_predictions.png"))
display(Image(filename=analysis.FIGURE_DIR / "aroma_parameter_comparison.png"))
display(Image(filename=analysis.FIGURE_DIR / "ethyl_octanoate_loss_dynamics.png"))
display(Image(filename=analysis.FIGURE_DIR / "ethyl_octanoate_partition_decomposition.png"))

## Identificabilidad

In [ ]:
display(result["validation"].query("forcing_source == 'dynamic_transfer_model' and fit_scope == 'all_data'").round(4))
display(Image(filename=analysis.FIGURE_DIR / "release_forcing_profile_likelihood.png"))

## Takeaways

La comparación relevante es el desempeño holdout con el mismo rCO₂, cambiando solamente el operador de volatilización. Una mejora no implica por sí sola identificabilidad completa: `kLa` puede saturarse cuando la corriente gaseosa ya sale cerca del equilibrio. Los límites de detección se incorporan como censura, no como ceros.

In [ ]:
print("Notebook ejecutado sin errores.")
print("Figuras embebidas:", len(result["figures"]))
print("Artefactos:", analysis.RESULTS_DIR.relative_to(ROOT))